In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [2]:
# Preprocessing: Tangani nilai NaN dengan string kosong

# Menampilkan jumlah nilai NaN di setiap kolom
print("Jumlah nilai NaN per kolom:")
print(df.isnull().sum())
print("\nPersentase nilai NaN per kolom:", (df.isnull().sum() / len(df)) * 100)

features = ['title', 'director', 'cast', 'country', 'listed_in', 'description']
for feature in features:
    df[feature] = df[feature].fillna('')

Jumlah nilai NaN per kolom:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

Persentase nilai NaN per kolom: show_id          0.000000
type             0.000000
title            0.000000
director        29.908028
cast             9.367549
country          9.435676
date_added       0.113546
release_year     0.000000
rating           0.045418
duration         0.034064
listed_in        0.000000
description      0.000000
dtype: float64


In [3]:
# Pada kolom 'casts' dilakukan pembersihan Nama Tokoh (menghapus spasi)
def clean_names(text):
    if isinstance(text, str):
        # 1. Ubah ke huruf kecil
        text = text.lower()
        # 2. Pisahkan berdasarkan koma untuk mendapatkan list nama
        names = text.split(',')
        # 3. Hapus spasi di dalam setiap nama (misal: "kirsten johnson" -> "kirstenjohnson")
        cleaned_names = [name.replace(" ", "") for name in names]
        # 4. Gabungkan kembali dengan spasi biasa antar nama tokoh
        return " ".join(cleaned_names)
    return ""

# Terapkan fungsi pembersihan ke kolom 'cast' dan 'director'
df['cast_cleaned'] = df['cast'].apply(clean_names)
df['director_cleaned'] = df['director'].apply(clean_names)

In [4]:
def get_random_movie_title(dataframe):
    """
    Fungsi untuk mengambil satu judul film secara acak dari dataset.
    """
    # Mengambil 1 sampel acak dari kolom 'title', lalu mengambil nilai string-nya
    random_title = dataframe['title'].sample(n=1).values[0]
    return random_title

In [5]:
# 3. Membuat 'Metadata Soup' (Menggabungkan semua fitur teks)
def combine_features(row):
    # Kolom title, country, listed_in, dan description tetap seperti biasa
    # Kolom director dan cast menggunakan yang sudah dibersihkan spacenya
    return (str(row['title']) + " " + 
            row['director_cleaned'] + " " + 
            row['cast_cleaned'] + " " + 
            str(row['country']) + " " + 
            str(row['listed_in']) + " " + 
            str(row['description']))

df['combined_features'] = df.apply(combine_features, axis=1)

# Pastikan teks menggunakan huruf kecil semua untuk konsistensi embeddings
df['combined_features'] = df['combined_features'].str.lower()


In [6]:
# 4. Membuat Word Embeddings menggunakan Sentence-Transformers
# Catatan: Jika belum install, jalankan di terminal: pip install sentence-transformers
from sentence_transformers import SentenceTransformer

# Menggunakan model yang ringan tapi sangat powerful untuk text similarity
model = SentenceTransformer('all-MiniLM-L6-v2')
# Mengubah teks gabungan menjadi matriks vektor (Embeddings)
embeddings = model.encode(df['combined_features'].tolist(), show_progress_bar=True)

C:\Users\user\AppData\Roaming\Python\Python312\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/276 [00:00<?, ?it/s]

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# 5. Menghitung Cosine Similarity Matrix
cosine_sim = cosine_similarity(embeddings, embeddings)

In [8]:
# 6. Fungsi Rekomendasi
def get_content_based_recommendations_for_user(user_history, embeddings_matrix, top_n=5):
    """
    user_history: List of dict berisi film yang disukai user
    embeddings_matrix: Matriks embedding dari seluruh catalog film (misal dari SBERT/USE)
    """
    user_movie_indices = []
    
    # 1. Cari indeks dari setiap film yang ada di riwayat user
    for item in user_history:
        try:
            idx = df[df['title'].str.lower() == item['title'].lower()].index[0]
            user_movie_indices.append(idx)
        except IndexError:
            print(f"Film '{item['title']}' tidak ditemukan di katalog.")
            
    if not user_movie_indices:
        return "Tidak ada film riwayat user yang cocok dengan katalog."
    
    # 2. Ambil vektor embedding untuk film-film yang disukai user
    user_movie_embeddings = embeddings_matrix[user_movie_indices]
    
    # 3. Agregasi: Hitung rata-rata vektor untuk menjadi "Vektor Profil User"
    # Menghasilkan 1 vektor berdimensi tetap yang merepresentasikan selera user
    user_profile_vector = np.mean(user_movie_embeddings, axis=0).reshape(1, -1)
    
    # 4. Hitung kemiripan antara Vektor Profil User dengan SEMUA film di katalog
    user_similarity_scores = cosine_similarity(user_profile_vector, embeddings_matrix)[0]
    
    # 5. Gabungkan skor kemiripan ke dataframe untuk sorting
    df_scores = df.copy()
    df_scores['similarity_score'] = user_similarity_scores
    
    # 6. Filter agar tidak merekomendasikan kembali film yang SUDAH ditonton user
    df_filtered = df_scores.drop(user_movie_indices)
    
    # 7. Urutkan dari skor tertinggi dan ambil Top-N
    recommendations = df_filtered.sort_values(by='similarity_score', ascending=False).head(top_n)
    
    return recommendations[['title', 'listed_in', 'description', 'similarity_score']]

In [9]:
# Contoh testing hasil rekomendasi user 
riwayat_user = [
    {'title': get_random_movie_title(df), 'rating': 5},
    {'title': get_random_movie_title(df), 'rating': 5},
    {'title': get_random_movie_title(df), 'rating': 5},
    {'title': get_random_movie_title(df), 'rating': 3},
    {'title': get_random_movie_title(df), 'rating': 4},
    {'title': get_random_movie_title(df), 'rating': 2},
    {'title': get_random_movie_title(df), 'rating': 2},
]

# Print judul film dari riwayat_user
print("Riwayat film user:")
for movie in riwayat_user:
    print("-", movie['title'])

rekomendasi = get_content_based_recommendations_for_user(riwayat_user, embeddings, top_n=5)

rekomendasi

Riwayat film user:
- Come Sunday
- To Wong Foo, Thanks for Everything! Julie Newmar
- Maria Bamford: The Special Special Special
- The Royal House of Windsor
- My Horrible Grandma
- BluffMaster!
- Indiana Jones and the Last Crusade


,title,listed_in,description,similarity_score
2463,Grandmother's Farm,"Comedies, Horror Movies, International Movies",A guys' getaway to an isolated farm in the des...,0.656982
632,Wanted,"Comedies, International Movies",Four seniors embark on misadventures after bre...,0.654959
2300,The Perfect Picture: Ten Years Later,"Comedies, Dramas, International Movies","From flawed husbands to shaky finances, new co...",0.652583
2379,An Evening with Beverly Luff Linn,"Comedies, Independent Movies",When an unhappily married woman discovers a ma...,0.643825
2438,Sorry To Disturb,"Dramas, International Movies","After losing his father, a genius yet troubled...",0.641420


Evaluasi Performansi SBERT

Untuk menguji kualitas sistem rekomendasi berbasis konten (Content-Based RS), kita bisa menggunakan metrik Precision, Recall, dan F1-Score.

Namun, ada satu tantangan dalam data teks/film: metrik ini membutuhkan Ground Truth (pilihan mutlak apakah rekomendasi itu "benar" atau "salah"). Karena kita tidak punya data riwayat transaksi pengguna (user clicks/ratings), cara standar untuk mengujinya secara mandiri adalah menggunakan kesamaan kategori (misalnya kolom listed_in atau genre) sebagai tolok ukur kebenaran.

Asumsi Pengujian: Jika film yang direkomendasikan memiliki minimal satu genre yang sama dengan film yang dicari, maka rekomendasi tersebut dianggap Relevan (1). Jika tidak ada genre yang sama, dianggap Tidak Relevan (0).

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# ==========================================
# 1. FUNGSI UNTUK MENGECEK RELEVANSI (GROUND TRUTH)
# ==========================================
def check_relevance(target_genres, recommended_genres):
    """
    Fungsi untuk mengecek apakah genre film rekomendasi beririsan dengan film target.
    Contoh: Target = 'Documentaries', Rekomendasi = 'Documentaries, International' -> Relevan
    """
    # Mengubah string genre menjadi set kata kunci agar bisa dicari irisannya
    target_set = set([g.strip().lower() for g in target_genres.split(',')])
    rec_set = set([g.strip().lower() for g in recommended_genres.split(',')])
    
    # Jika ada kesamaan genre, kembalikan 1 (Relevan), jika tidak kembalikan 0
    return 1 if len(target_set.intersection(rec_set)) > 0 else 0


# ==========================================
# 2. FUNGSI EVALUASI UNTUK SATU JUDUL FILM
# ==========================================
def evaluate_recommendations(title, cosine_sim_matrix=cosine_sim, k=5):
    # Ambil indeks dan genre film target
    try:
        idx = df[df['title'].str.lower() == title.lower()].index[0]
        target_genres = df.loc[idx, 'listed_in']
    except IndexError:
        print(f"Judul '{title}' tidak ditemukan.")
        return None
    
    # Ambil Top-K rekomendasi (dari fungsi rekomendasi sebelumnya)
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:k+1] # Ambil K film teratas (misal 5)
    
    movie_indices = [i[0] for i in sim_scores]
    recommended_movies = df.iloc[movie_indices]
    
    # Hitung ground truth kebenaran (Aktual) vs Prediksi kita
    # Karena model merekomendasikannya, maka y_prediksi semuanya bernilai 1 (Sistem menebak ini relevan)
    y_pred = [1] * k
    
    # Cek apakah film yang direkomendasikan aktualnya relevan berdasarkan genre
    y_true = []
    for _, row in recommended_movies.iterrows():
        relevance = check_relevance(target_genres, row['listed_in'])
        y_true.append(relevance)
        
    # Hitung metrik evaluasi
    # Menggunakan zero_division=0 untuk menghindari error jika tidak ada yang relevan sama sekali
    precision = precision_score(y_true, y_pred, zero_division=0)
    
    # Catatan: Pada Top-K rekomendasi tanpa batasan total item relevan, 
    # nilai Recall sering kali sama dengan Precision karena jumlah prediksi (K) selalu tetap.
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    return {
        'Movie': title,
        'Precision@K': precision,
        'Recall@K': recall,
        'F1-Score@K': f1,
        'Detail Rekomendasi (Relevansi Aktual)': y_true
    }

--- Hasil Evaluasi Single Movie ---
Movie: Blood & Water
Precision@K: 1.0
Recall@K: 1.0
F1-Score@K: 1.0
Detail Rekomendasi (Relevansi Aktual): [1, 1, 1, 1, 1]


In [ ]:
# Mengambil 10 film secara acak dari dataset untuk dijadikan bahan uji coba
sample_movies = df['title'].sample(50, random_state=42).tolist()

all_precision = []
all_recall = []
all_f1 = []

for movie in sample_movies:
    res = evaluate_recommendations(movie, k=5)
    if res:
        all_precision.append(res['Precision@K'])
        all_recall.append(res['Recall@K'])
        all_f1.append(res['F1-Score@K'])

print("--- HASIL EVALUASI RATA-RATA (10 Sampel Film) ---")
print(f"Mean Precision@5 : {np.mean(all_precision):.2f}")
print(f"Mean Recall@5    : {np.mean(all_recall):.2f}")
print(f"Mean F1-Score@5  : {np.mean(all_f1):.2f}")


--- HASIL EVALUASI RATA-RATA (10 Sampel Film) ---
Mean Precision@5 : 0.72
Mean Recall@5    : 0.92
Mean F1-Score@5  : 0.79
